# Theorem 2 positive companion: artefact-learning can win

This notebook builds a **positive Theorem 2** companion task: a small `2×n` tiling-style artifact task where the **finished artifact already contains the relevant structure**, so exposing failed attempts can act as noise rather than help.

The workflow compares:
- `artifact_only`: train on completed successful tilings only
- `mixed_failed`: train on corpora that include failed attempts before the final solution

Primary question: does the artifact-only environment produce better **pure successful artifact generation**?


In [1]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
for p in [HERE, *HERE.parents]:
    if p.name == "Drift_and_selection" or ((p / "GitHub").exists() and (p / "Nat_Paper").exists()):
        PROJECT_ROOT = p if p.name == "Drift_and_selection" else p
        break
else:
    PROJECT_ROOT = HERE

SRC = PROJECT_ROOT / "GitHub" / "src" / "drift_selection"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if "/mnt/data" not in sys.path:
    sys.path.insert(0, "/mnt/data")

PROJECT_ROOT, SRC


(PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection'),
 PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/src/drift_selection'))

In [2]:
from theorem2_artifact_learning import (
    TilingTaskConfig, ModelConfig, TrainConfig,
    build_tiling_datasets, pretty_print_examples,
    run_tiling_pipeline, default_output_roots
)

ROOTS = default_output_roots(PROJECT_ROOT)
ROOTS


{'data_out': PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/data/outputs/theorem2_artifact_learning'),
 'fig_out': PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/figures/appendix/theorem2_artifact_learning'),
 'nb_out': PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/notebooks/active'),
 'src_out': PosixPath('/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/src/drift_selection')}

## Configuration

Defaults are pilot-friendly but large enough to show a signal. Training widths are smaller than test widths so the task probes some compositional generalization.


In [3]:
RUN_BUILD_PREVIEWS = True
RUN_PIPELINE = True
RUN_PROFILE = "bridge_mild_ood"  # or "sanity_in_dist"

# Profiles:
# - sanity_in_dist: easiest, train and test on same width range
# - bridge_mild_ood: mild extrapolation
# - hard_ood: strongest extrapolation
PROFILE_SPECS = {
    "sanity_in_dist": {
        "train_min_n": 6,
        "train_max_n": 10,
        "test_min_n": 6,
        "test_max_n": 10,
        "n_train": 12000,
        "n_val": 1500,
        "n_test": 3000,
        "n_failed_attempts": 4,
        "epochs": 12,
    },
    "bridge_mild_ood": {
        "train_min_n": 6,
        "train_max_n": 10,
        "test_min_n": 10,
        "test_max_n": 12,
        "n_train": 12000,
        "n_val": 1500,
        "n_test": 3000,
        "n_failed_attempts": 2,
        "epochs": 10,
    },
    "hard_ood": {
        "train_min_n": 6,
        "train_max_n": 10,
        "test_min_n": 11,
        "test_max_n": 14,
        "n_train": 12000,
        "n_val": 1500,
        "n_test": 3000,
        "n_failed_attempts": 4,
        "epochs": 12,
    },
}

if RUN_PROFILE not in PROFILE_SPECS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}. Choose one of {list(PROFILE_SPECS)}")

spec = PROFILE_SPECS[RUN_PROFILE]

TASK_CFG = TilingTaskConfig(
    train_min_n=spec["train_min_n"],
    train_max_n=spec["train_max_n"],
    test_min_n=spec["test_min_n"],
    test_max_n=spec["test_max_n"],
    n_train=spec["n_train"],
    n_val=spec["n_val"],
    n_test=spec["n_test"],
    seed=123,
    n_failed_attempts=spec["n_failed_attempts"],
    equalize_examples_to_longest=True,
    deterministic_success=True,
)

MODEL_CFG = ModelConfig(
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=512,
    dropout=0.1,
    max_seq_len=256,
)

TRAIN_CFG = TrainConfig(
    batch_size=64,
    learning_rate=3e-4,
    weight_decay=0.01,
    epochs=spec["epochs"],
    clip_grad_norm=1.0,
    seed=123,
)

STYLES = ["artifact_only", "mixed_failed"]
RUN_NAME = f"theorem2_positive_tiling_artifact_learning__{RUN_PROFILE}_ep{TRAIN_CFG.epochs}"

TASK_CFG, MODEL_CFG, TRAIN_CFG, RUN_NAME


(TilingTaskConfig(train_min_n=6, train_max_n=10, test_min_n=10, test_max_n=12, n_train=12000, n_val=1500, n_test=3000, seed=123, n_failed_attempts=2, equalize_examples_to_longest=True, deterministic_success=True),
 ModelConfig(d_model=128, n_heads=4, n_layers=2, d_ff=512, dropout=0.1, max_seq_len=256),
 TrainConfig(batch_size=64, learning_rate=0.0003, weight_decay=0.01, epochs=10, clip_grad_norm=1.0, eval_every_epoch=True, seed=123, checkpoint_every_epoch=True),
 'theorem2_positive_tiling_artifact_learning__bridge_mild_ood_ep10')

In [4]:
datasets = build_tiling_datasets(STYLES, TASK_CFG)

if RUN_BUILD_PREVIEWS:
    print("ARTIFACT-ONLY examples")
    print(pretty_print_examples(datasets["artifact_only"]["train"], n=3))
    print("MIXED-FAILED examples")
    print(pretty_print_examples(datasets["mixed_failed"]["train"], n=3))

{style: {split: len(rows) for split, rows in splits.items()} for style, splits in datasets.items()}


ARTIFACT-ONLY examples
PROMPT: TASK TILE N 6 SOLVE
TARGET: SOL HH HH HH

PROMPT: TASK TILE N 8 SOLVE
TARGET: SOL HH HH HH HH

PROMPT: TASK TILE N 6 SOLVE
TARGET: SOL HH HH HH

MIXED-FAILED examples
PROMPT: TASK TILE N 6 SOLVE
TARGET: TRY HH V HH HH FAIL OVER TRY V HH SQ SQ FAIL OVER SOL HH HH HH

PROMPT: TASK TILE N 8 SOLVE
TARGET: TRY HH V REM 5 FAIL SHORT TRY SQ REM 6 FAIL SHORT SOL HH HH HH HH

PROMPT: TASK TILE N 6 SOLVE
TARGET: TRY HH V SQ HH FAIL OVER TRY V HH SQ HH FAIL OVER SOL HH HH HH



{'artifact_only': {'train': 48985, 'val': 6152, 'test': 3000},
 'mixed_failed': {'train': 12000, 'val': 1500, 'test': 3000}}

In [5]:
def token_budget_summary(ds):
    out = {}
    for style, splits in ds.items():
        out[style] = {}
        for split, rows in splits.items():
            out[style][split] = sum(len(r["target_tokens"]) for r in rows)
    return out

token_budget_summary(datasets)


{'artifact_only': {'train': 254455, 'val': 31900, 'test': 19956},
 'mixed_failed': {'train': 254455, 'val': 31900, 'test': 74692}}

## Run the full artifact-learning pipeline

This will:
1. save the generated datasets
2. train one model per style with checkpoint/resume
3. evaluate on held-out larger-width tiling prompts
4. save summary metrics, predictions, samples, and training curves


In [6]:
if RUN_PIPELINE:
    result = run_tiling_pipeline(
        styles=STYLES,
        task_cfg=TASK_CFG,
        model_cfg=MODEL_CFG,
        train_cfg=TRAIN_CFG,
        run_name=RUN_NAME,
        project_root=PROJECT_ROOT,
    )
    print(result["run_root"])
    display(result["metrics_df"])
else:
    print("Set RUN_PIPELINE = True to train and evaluate the positive Theorem 2 companion task.")


train:artifact_only: 0epoch [00:00, ?epoch/s]

train:mixed_failed: 0epoch [00:00, ?epoch/s]

eval-tiling:   0%|          | 0/3000 [00:00<?, ?it/s]

eval-tiling:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/sorenriis/Documents/ChatGpt_codex_folder/Drift_and_selection/GitHub/data/outputs/theorem2_artifact_learning/theorem2_positive_tiling_artifact_learning__bridge_mild_ood_ep10


,style,task_family,valid_last_solution_rate,pure_valid_solution_rate,failure_marker_rate
0,artifact_only,tiling_artifact_learning,0.348,0.348,0.0
1,mixed_failed,tiling_artifact_learning,0.348,0.000,1.0


In [7]:
# If you have run the pipeline, this cell will load the summary metrics CSV for inspection.
from pathlib import Path
import pandas as pd
run_root = ROOTS["data_out"] / RUN_NAME
metrics_csv = run_root / "evaluation" / "summary_metrics.csv"
if metrics_csv.exists():
    display(pd.read_csv(metrics_csv))
else:
    print("No metrics CSV yet:", metrics_csv)
    if not RUN_PIPELINE:
        print("Set RUN_PIPELINE = True in the config cell, then run the pipeline cell.")
    else:
        print("Pipeline may still be running, or it ended early; check notebook outputs above.")


,style,task_family,valid_last_solution_rate,pure_valid_solution_rate,failure_marker_rate
0,artifact_only,tiling_artifact_learning,0.348,0.348,0.0
1,mixed_failed,tiling_artifact_learning,0.348,0.000,1.0
